# 04 Extended model: RFI injection and mitigation

Injects RFI into the Sentinel-1 test inputs and evaluates the variants
M0 (clean), M1 (contaminated), M2 (published suppression, not available),
M3 (detect-and-fill), M3b (detect-and-discard) and M4 (detect-and-fill with
RFI-aware training) for the feature configurations F1, F2 and F3 and the
change-detection baseline. The injection grid comes from the
OpenSAR statistics (severity and extent levels from 125 patches). All logic is in `src/rfi.py` and
`src/experiment.py`. A full run takes about 70 minutes on four workers.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

In [ ]:
from src.experiment import run_stage3

out = run_stage3()

## Degradation and recovery by variant (all settings)

D = RMSE_M1 - RMSE_M0, G = RMSE_M1 - RMSE_variant, RR = G / D, in m3/m3,
averaged over settings and seeds. M2 rows are empty (not available).

In [ ]:
cols = ["evaluation", "feature_set", "model", "variant", "rmse_M0", "rmse_M1",
        "rmse_variant", "D_mean", "G_mean", "RR_of_means", "RR_median"]
out["stage3_summary_overall"][cols].round(4)

## Degradation by severity level and by extent, variant M3 rows

In [ ]:
s = out["stage3_summary_by_severity"]
s[s.variant == "M3"].pivot_table(index=["evaluation", "feature_set", "model"],
                                 columns="severity_q", values="D_mean").round(4)

In [ ]:
s = out["stage3_summary_by_extent"]
s[s.variant == "M3"].pivot_table(index=["evaluation", "feature_set", "model"],
                                 columns="extent", values="D_mean").round(4)

## Detection precision, recall and F1 by severity level

In [ ]:
out["stage3_detection_by_severity"].round(3)